In [1]:
"""
This script filters fragment files, based on their condition in another (metadata) table
authors: Roy Oelen
"""

'\nThis script is used regress out coeqtl PCs\nauthors: Roy Oelen\n'

In [2]:
# required libraries
import numpy as np
import gzip
import pandas as pd

/local/11206/ipykernel_768034/648356095.py:4: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [3]:

def split_fragments_on_barcodes(fragments_loc, output_loc, metadata_table, lane, lane_column='lane', split_column='condition_final', barcode_column='barcode_1'):
    # get the barcodes that are of this lane
    metadata_lane = metadata_table[metadata_table[lane_column] == lane]
    # extract the unique splits
    split_possibilities = metadata_lane[split_column].unique().tolist()
    # create the output file handles for each 
    handle_per_split = {}
    for split in split_possibilities:
        # get the full output location
        split_out = ''.join([output_loc, '', lane, '_', split, '.tsv.gz'])
        # open a filehandle
        filehandle_split = gzip.open(split_out, 'wt')
        # put in a the dictionary
        handle_per_split[split] = filehandle_split
    # make a dictionary of the barcode and the split
    barcode_to_split = dict(zip(metadata_lane[barcode_column], metadata_lane[split_column]))
    # now go through the fragment file
    with gzip.open(fragments_loc,'rt') as f:
        for line in f:
            # check if we are at the header
            if line.startswith('#'):
                # we need to write that to all files
                for handle_split in handle_per_split.values():
                    handle_split.write(line)
            else:
                # remove newline character
                clean_line = line.rstrip('\r\n')
                # split file by tab
                values_line = clean_line.split('\t')
                # get the barcode
                barcode = values_line[3]
                # check if we have this key
                if barcode in barcode_to_split:
                    # get the split for that barcode
                    barcode_split = barcode_to_split[barcode]
                    # write to the correct split file
                    handle_split = handle_per_split[barcode_split]
                    handle_split.writelines(line)
                
    # close the file handle
    for handle_split in handle_per_split.values():
        handle_split.close()

In [4]:
# location of the metadata file
metadata_loc = '/groups/umcg-franke-scrna/tmp03/projects/multiome/ongoing/metadata/mo_celllevel_metadata.tsv.gz'
# read the metadata
metadata = pd.read_csv(metadata_loc, sep = '\t')

/local/11206/ipykernel_768034/1825496088.py:4: DtypeWarning: Columns (15,20,36,51) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(metadata_loc, sep = '\t')


In [5]:
split_fragments_on_barcodes('/groups/umcg-franke-scrna/tmp03/projects/multiome/processed/joint/alignment/b38/230302_lane2/outs/atac_fragments.tsv.gz',
                            '/groups/umcg-franke-scrna/tmp03/projects/multiome/ongoing/cpeaks_fragment_overlap/per_lane/230302_lane2/', 
                            metadata, 
                            '230302_lane2')